# Reranking Matrix Explorer
    
Inspect the stored reranking comparison matrices located under `data/external/reranking-matrices`.
    


## Setup
    
Run the next cell to discover available matrices. Adjust the later `TARGET_FILE` variable to inspect a specific one.
    


In [2]:
from pathlib import Path
import pickle
from typing import Dict, Tuple
from collections import Counter

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

BASE = Path("../data/external/reranking-matrices")
if not BASE.exists():
    raise FileNotFoundError(f"Expected folder {BASE} not found")

files = sorted(BASE.rglob("*.pkl"))
print(f"Discovered {len(files)} matrix files under {BASE}")


Discovered 24 matrix files under ../data/external/reranking-matrices


## Dataset summary
    
The following cell loads each matrix and reports the number of queries and comparisons stored.
    


In [3]:
def dataset_from_name(path: Path) -> str:
    stem = path.stem
    parts = stem.split("_")
    if len(parts) >= 2:
        return parts[-1]
    return stem

summary = []
for path in files:
    try:
        with path.open("rb") as f:
            data = pickle.load(f)
        qids = {k[0] for k in data if isinstance(k, tuple) and len(k) == 3}
        summary.append(
            {
                "file": str(path.relative_to(BASE)),
                "dataset": dataset_from_name(path),
                "pairs": len(data),
                "unique_queries": len(qids),
            }
        )
    except Exception as exc:
        summary.append(
            {
                "file": str(path.relative_to(BASE)),
                "dataset": dataset_from_name(path),
                "pairs": None,
                "unique_queries": None,
                "error": str(exc),
            }
        )

if pd is not None:
    display(pd.DataFrame(summary))
else:
    for row in summary:
        print(row)


,file,dataset,pairs,unique_queries
0,Reranking/2024_10_12_13_27_22_flan-t5-xl_webis...,webis-touche2020,485100,49
1,Reranking/2024_10_13_08_34_48_Meta-Llama-3-8B-...,dl-2020,1980000,200
2,Reranking/inferences/flan-t5-xl_trec-covid.pkl,trec-covid,495000,50
3,Reranking/large/2024_09_23_19_33_41_flan-t5-la...,webis-touche2020,485100,49
4,Reranking/large/2024_09_23_23_44_56_flan-t5-la...,trec-news,564300,57
5,Reranking/large/2024_09_24_12_23_41_flan-t5-la...,nfcorpus,2077642,295
6,Reranking/large/2024_09_24_15_28_31_flan-t5-la...,trec-covid,495000,50
7,Reranking/large/2024_09_25_10_11_04_flan-t5-la...,scifact,2964390,300
8,Reranking/large/2024_09_27_10_08_08_flan-t5-la...,fiqa,6415200,648
9,Reranking/large/2024_09_30_08_00_10_flan-t5-la...,robust04,2465100,249


## Choose a matrix to inspect
    
Set `TARGET_FILE` to one of the `files` entries reported above.
    


In [ ]:
if not files:
    TARGET_FILE = "data/external/reranking-matrices/Reranking/large/2024_09_23_19_33_41_flan-t5-large_webis-touche2020.pkl"
    print("No matrix files available")
else:
    TARGET_FILE = files[0]
    print(f"Inspecting {TARGET_FILE}")


Inspecting ../data/external/reranking-matrices/Reranking/2024_10_12_13_27_22_flan-t5-xl_webis-touche2020.pkl


In [9]:
if TARGET_FILE is None:
    raise RuntimeError("Set TARGET_FILE to a valid path from the 'files' list")

with TARGET_FILE.open("rb") as f:
    matrix = pickle.load(f)
print(f"Loaded {len(matrix)} pairwise entries from {TARGET_FILE}")

import itertools

sample_items = list(itertools.islice(matrix.items(), 5))  # first 5 pairs
for (qid, doc_a, doc_b), entry in sample_items:
    print(qid, doc_a, doc_b, "->", entry)


Loaded 485100 pairwise entries from ../data/external/reranking-matrices/Reranking/2024_10_12_13_27_22_flan-t5-xl_webis-touche2020.pkl
37 28b1a24d-2019-04-18T19:06:02Z-00000-000 7cb4f1d5-2019-04-18T16:18:51Z-00001-000 -> {'text': 'Passage A', 'scores': [('A', np.float16(0.6807)), ('B', np.float16(0.0588))]}
1 e0ccca84-2019-04-18T19:11:07Z-00005-000 5a48ffd9-2019-04-18T18:37:49Z-00001-000 -> {'text': 'Passage A', 'scores': [('A', np.float16(0.1807)), ('B', np.float16(-3.1))]}
46 2b6ac2c3-2019-04-18T15:10:44Z-00001-000 2b6ac301-2019-04-18T11:55:15Z-00000-000 -> {'text': 'Passage A', 'scores': [('A', np.float16(0.915)), ('B', np.float16(-1.564))]}
4 dc1d3e0b-2019-04-18T19:21:17Z-00000-000 2fc6200f-2019-04-18T17:01:39Z-00003-000 -> {'text': 'Passage B', 'scores': [('A', np.float16(0.3618)), ('B', np.float16(1.101))]}
10 f083f25a-2019-04-18T19:14:24Z-00005-000 9f71c585-2019-04-18T19:02:47Z-00004-000 -> {'text': 'Passage B', 'scores': [('A', np.float16(0.2947)), ('B', np.float16(1.33))]}


## Query-level inspection
    
Provide a query ID below to inspect all pairs associated with it. When `QUERY_ID` is `None`, the notebook picks the first available query.
    


In [6]:
if TARGET_FILE is None:
    raise RuntimeError("TARGET_FILE must be set before inspecting queries")

QUERY_ID = None  # e.g., "37"
if QUERY_ID is None:
    QUERY_ID = next(iter({k[0] for k in matrix if isinstance(k, tuple)}))

pairs_for_query = {
    (doc_a, doc_b): matrix[key]
    for key in matrix
    if isinstance(key, tuple) and len(key) == 3 and key[0] == QUERY_ID
    for doc_a, doc_b in [(key[1], key[2])]
}
print(f"Query {QUERY_ID} has {len(pairs_for_query)} comparisons")
sample = list(pairs_for_query.items())[:5]
for (doc_a, doc_b), entry in sample:
    print("-", doc_a, "vs", doc_b, "=>", entry.get("text"))


Query 31 has 9900 comparisons
- e4feb1b5-2019-04-18T12:43:31Z-00004-000 vs 21e2b85c-2019-04-18T15:17:47Z-00002-000 => Passage B
- 3c486dec-2019-04-18T16:40:15Z-00004-000 vs 36aea26b-2019-04-18T18:57:44Z-00005-000 => Passage A
- d766d928-2019-04-18T18:00:40Z-00004-000 vs bbc5b97a-2019-04-18T16:48:53Z-00000-000 => Passage A
- 962b00f0-2019-04-18T17:46:15Z-00005-000 vs 636669d7-2019-04-18T19:49:10Z-00007-000 => Passage B
- fb2e263c-2019-04-18T19:34:26Z-00000-000 vs c4eaae9b-2019-04-18T15:22:52Z-00001-000 => Passage B
